# Part 2 — MOFA tools over MCP (FastMCP + langchain-mcp-adapters)

In this notebook, we learn about **MCP (the Model Context Protocol)**: a standard way to serve tools from a server so that any agent can call them without needing your code.

In this notebook, we'll cover two things:
1. **Calling LLM tools through the MCP**: We'll take the exact same tools from Part 1 and serve them through an MCP server (`server/mofa_mcp_server.py`). This shows the mechanics: how tools move from local to a shared server.
2. **Bringing in an external MCP server — BioMCP**: We'll then connect to BioMCP, a community-built MCP server that queries over 40 biomedical data sources (PubMed, UniProt, ClinVar, and others) through a single interface, showing how MCPs eliminate the need to learn different APIs—one standard protocol connects you to many tools.

## Learning objectives

By the end of this notebook you should be able to:
- Explain what MCP is, and why it changes where a tool lives and who can reach it, not what the tool can do.
- Serve the same tools from Part 1 through an MCP server, and confirm Claude gets the identical answer through it.
- Explain the two reasons MCP is worth the extra layer: keeping data and logic in one controlled place, and reusing tools other people have already built and published.
- Connect to a public MCP server (BioMCP) alongside your own, and write a system prompt that tells Claude when to use which.

### Model backend: bring your own key

Reads `ANTHROPIC_API_KEY` from `.env`; run with the `eccb` kernel. Only the agent-run cells later in the notebook spend API tokens — everything above them is local: the server subprocess loads the cached model, no LLM involved.

### Why MCP here?

For this practical, the MCP server buys little over the plain functions in Part 1, and that is the point: the task is identical, so the only thing that changes is the interface. MCP starts to pay off when the tools front something you do not want inside every notebook process — here, that is a 1.1 GB `omics.pkl` and a fitted model. The server loads that once at startup and exposes only small, audited, read-only operations; many clients can reuse it without ever loading that data themselves.

## 0. Environment & API key

Load the key, and point the notebook at the project root and the MCP server script.

In [ ]:
from pathlib import Path
import os, sys, json

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic


In [ ]:
# claude_notebooks/ sits one level deeper than notebooks/ (notebooks/claude_notebooks/),
# so it needs to climb two levels instead of one to reach the project root.
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY not found. Add it to a .env in the project root.")
print("ANTHROPIC_API_KEY loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

SERVER_PATH = PROJECT_ROOT / "server" / "mofa_mcp_server.py"
print("MCP server:", SERVER_PATH.name)


## 1. The MCP server

`server/mofa_mcp_server.py` uses **FastMCP** (a Python library for building MCP servers with minimal boilerplate) to expose three MCP primitives, all backed by the cached MOFA model:

- **Tools** (model-callable, read-only): `data_summary`, `split_summary`, `active_factors`, `factor_view_r2`, `factor_subtype_association`, `top_features_for_factor`, `classify_subtype_from_factors`, `train_vs_test_subtype_association`
- **Resource**: `mofa://summary` — a compact model/cohort summary to read into context
- **Prompt**: `interpret_factor(factor)` — a reusable interpretation template

The server loads the aligned omics and fitted model once at startup; `fit_mofa` is not exposed here either, for the same reason as Part 1. The clients below launch it as a `stdio` subprocess — a subprocess your notebook talks to over its standard input/output, the same way you would pipe data between two command-line programs.

## 2. Under the hood: talk to MCP directly (no adapter)

Before bringing in the LangChain adapter, we connect with MCP's own client library directly, to see what MCP actually is underneath the convenience wrapper: open a `ClientSession` over `stdio`, ask the server what it offers (*handshake*), then call one tool over the wire. No LLM involved, this costs nothing to run.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        init = await session.initialize()
        print(f"Connected to: {init.serverInfo.name}")
        caps = init.capabilities
        print("Server advertises:",
            "tools" if caps.tools else "", "resources" if caps.resources else "",
            "prompts" if caps.prompts else "")

        print("\n TOOLS (the model may choose to call these)")
        for t in (await session.list_tools()).tools:
            args = ", ".join(t.inputSchema.get("properties", {}))
            ro = getattr(t.annotations, "readOnlyHint", None)
            print(f"  {t.name}({args})   readOnlyHint={ro}")

        print("\n RESOURCES (data to read into context) ")
        for r in (await session.list_resources()).resources:
            print(f"  {r.uri}  ({r.name})")
        summary = await session.read_resource("mofa://summary")
        print("  read mofa://summary ->", summary.contents[0].text[:80].replace(chr(10), " "), "...")

        print("\n PROMPTS (server-provided templates) ")
        for p in (await session.list_prompts()).prompts:
            args = ", ".join(a.name for a in (p.arguments or []))
            print(f"  {p.name}({args})")

        print("\n tools/call factor_subtype_association")
        called = await session.call_tool("factor_subtype_association", {})
        print("  ", called.content[0].text[:120].replace(chr(10), " "))


The block above never imported LangChain — that was MCP directly.

MCP messages travel as **JSON-RPC** (a simple request/response message format) over a transport — here that is `stdio`, though it could be HTTP instead. On that wire, MCP defines a fixed set of methods: `initialize`, `tools/list`, `tools/call`, `resources/list`, `resources/read`, `prompts/list`, `prompts/get`.

Tools are only one of several things a server can offer:

| Primitive | Driven by | What it is | In our server |
|-----------|-----------|------------|---------------|
| **Tools** | the model | functions the model may choose to call | the 8 analysis functions |
| **Resources** | the app/user | data to read into context | `mofa://summary` |
| **Prompts** | the user | reusable, parameterized templates | `interpret_factor` |
| **Sampling** | the server | server asks the client's LLM to generate | (not used) |
| **Elicitation** | the server | server asks the human mid-call | (not used) |
| **Roots** | the client | filesystem/scope boundaries | (not used) |

Each tool also carried `readOnlyHint=True` — metadata a client could use to auto-run a tool instead of pausing for human approval, which matters specifically because the caller here is a model, not a fixed script. A generic remote-procedure-call setup has no equivalent notion.

One more useful pattern before we bring in the adapter: wrap a single call in a small reusable function, the way `MCPTool` will do more fully further down.

In [ ]:
async def get_resource(uri):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.read_resource(uri)

res = await get_resource("mofa://summary")
print(res.contents[0].text)


## 3. Framework adaptation: `langchain-mcp-adapters`

We call `get_tools()`, so only the *tools* primitive flows into the agent. The same client also exposes `get_resources()` and `get_prompt()`. Nothing about MCP is thrown away here — the adapter **is** itself an MCP client, speaking the identical wire protocol shown above; `get_tools()` just converts the one primitive a tool-calling loop actually needs.

## 4. Sanity-check one tool (no LLM)

Call a tool straight through the adapter's building blocks so the structured evidence is visible before involving the model at all.

In [ ]:
def parse_mcp(result):
    """MCP tools return text content blocks; json.loads each back to a dict/list."""
    if isinstance(result, list):
        out = []
        for block in result:
            text = block.get("text") if isinstance(block, dict) else block
            parsed = json.loads(text) if isinstance(text, str) else text
            out.append(_unwrap(parsed))
        return out
    parsed = json.loads(result) if isinstance(result, str) else result
    return _unwrap(parsed)

def _unwrap(parsed):
    """Undo the {"result": [...]} auto-wrapping MCP applies to non-object tool outputs."""
    if isinstance(parsed, dict) and list(parsed.keys()) == ["result"]:
        return parsed["result"]
    return parsed

class MCPTool:
    def __init__(self, name):
        self.name = name

    async def ainvoke(self, arguments):
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(self.name, arguments)
                parsed = [json.loads(block.text) for block in result.content]
                # if the tool returns a single object rather than a list of records,
                # collapse back down to that one object instead of a 1-item list
                if len(parsed) == 1:
                    return parsed[0]
                return parsed


In [ ]:
tools_by_name = {
    "factor_subtype_association": MCPTool("factor_subtype_association"),
    "classify_subtype_from_factors": MCPTool("classify_subtype_from_factors"),
}

assoc = await tools_by_name["factor_subtype_association"].ainvoke({})
print("factor_subtype_association (top 3):", assoc[:3])

cls = await tools_by_name["classify_subtype_from_factors"].ainvoke({})
print("classify_subtype_from_factors    :", cls)


## 5. Bind the MCP tools to Claude

Identical to Part 1: `bind_tools` does not care where the tools came from. Binding does not call the API.

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import create_model

async def _list_mcp_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return (await session.list_tools()).tools

mcp_tool_specs = await _list_mcp_tools()

def _make_langchain_tool(spec, mcp_tool: MCPTool):
    async def _run(**kwargs):
        return await mcp_tool.ainvoke(kwargs)
    return StructuredTool.from_function(
        coroutine=_run,
        name=spec.name,
        description=spec.description or spec.name,
        args_schema=None,  # LangChain will infer a permissive schema; fine for demo use
    )

tools = [
    _make_langchain_tool(spec, MCPTool(spec.name))
    for spec in mcp_tool_specs
]
tools_by_name = {t.name: t for t in tools}


In [ ]:
MODEL = "claude-haiku-4-5"
llm = ChatAnthropic(model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)
print("Bound", len(tools), "MCP tools to", MODEL)


## 6. Run the agent loop (async)

Same loop as Part 1, but `await`-ing the model and the tools, since MCP tools are asynchronous here.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

SYSTEM = ("You are a computational-biology assistant analysing a fitted MOFA model of "
          "TCGA breast-cancer multi-omics data. Use the tools to gather evidence; ground "
          "every quantitative claim in tool results and name which tool you used. Factors "
          "are 'Factor1'..'Factor10'; subtypes are PAM50 (LumA, LumB, Basal, Her2, Normal).")

async def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    for step in range(max_steps):
        ai = await llm_with_tools.ainvoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai
        for call in ai.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = await tools_by_name[call["name"]].ainvoke(call["args"])
            messages.append(ToolMessage(content=json.dumps(result, default=str),
                                        tool_call_id=call["id"]))
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")


The tool wrapping above is deliberately loose — `args_schema=None` means LangChain has to guess how to validate Claude's arguments. The version below fixes that by converting each tool's real MCP schema into a proper Pydantic model, so a bad argument gets caught before it ever reaches the server.

In [ ]:
from typing import Any
from pydantic import create_model

def _json_schema_to_pydantic(name: str, input_schema: dict):
    """Turn an MCP tool's JSON schema into a Pydantic model LangChain can use."""
    props = input_schema.get("properties", {})
    required = set(input_schema.get("required", []))
    type_map = {"string": str, "integer": int, "number": float, "boolean": bool}

    fields = {}
    for prop_name, prop_spec in props.items():
        py_type = type_map.get(prop_spec.get("type"), Any)
        default = ... if prop_name in required else prop_spec.get("default", None)
        fields[prop_name] = (py_type, default)

    return create_model(f"{name}_Args", **fields)


def _make_langchain_tool(spec, mcp_tool: MCPTool):
    ArgsModel = _json_schema_to_pydantic(spec.name, spec.inputSchema)

    async def _run(**kwargs):
        return await mcp_tool.ainvoke(kwargs)

    return StructuredTool.from_function(
        coroutine=_run,
        name=spec.name,
        description=spec.description or spec.name,
        args_schema=ArgsModel,
    )


In [ ]:
tools = [_make_langchain_tool(spec, MCPTool(spec.name)) for spec in mcp_tool_specs]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)


In [ ]:
final = await run_agent(
    "Which MOFA factor is most associated with breast-cancer subtype, and which "
    "transcriptomic features most strongly drive it?")
print(final.content)


## 7. Same answer as the direct-tools track

The flagship answer should match Part 1 — same backend and cached model, different transport. Compare the factor Claude picked against the direct evidence:

In [ ]:
print("Direct evidence (top subtype-associated factors):")
for row in assoc[:3]:
    print(f"  {row['factor']}: eta_squared = {row['eta_squared']}")


## Reflection

- Primitives: name the three this server exposes. Which adapter calls load the resource and the prompt instead of the tools? (`get_resources()`, `get_prompt()`)
- Did the adapter throw MCP away? No — it is an MCP client speaking the Section 2 protocol. What actually differs between Sections 2 and 5? (convenience and object model, not capability.)
- Layering: MCP rode on `stdio` + JSON-RPC. What changes if the transport is HTTP? (the primitives do not.)
- Annotations: every tool was `readOnlyHint=True`. How would a client use that? What changes for a tool that *writes*?
- Same answer: it matched Part 1. So what did MCP buy here, and when does it pay off? (big/private data behind an audited server, reuse across clients.)

## 8. Bringing in an external MCP server: BioMCP

So far our only MCP server has been our own: a private, local server fronting our own fitted model and data. That is one end of the spectrum. The other end: public, third-party MCP servers that expose general biomedical knowledge — we do not own or run them, we just connect to them as a client, the same way we connect to `mofa_mcp_server.py`.

[BioMCP](https://biomcp.org) is a real example: an open-source MCP server that fronts roughly 15 public biomedical data sources (PubMed, ClinicalTrials.gov, MyGene.info, MyChem.info, MyDisease.info, ClinVar, and others) behind a consistent set of tools — gene/drug/disease lookups, literature search, trial search, and more.

Why this matters pedagogically: it is the same MCP client code, connecting to a completely different kind of server — one we did not write, running somewhere else, fronting public knowledge instead of our private model. The client only needs to speak MCP; it does not need to know who runs the server or what is behind it.

### Install BioMCP

```bash
pip install biomcp-python
# or, for the newer single-binary CLI: uv tool install biomcp-cli
```

### Two servers, one agent

We now give the agent **two** MCP servers at once: our private `mofa` server
(local data) and the public `biomcp` server (general biomedical knowledge).
`MultiServerMCPClient` supports this natively — just add a second entry.

In [ ]:
import shutil, subprocess, sys
from langchain_mcp_adapters.client import MultiServerMCPClient

BIOMCP_BIN = shutil.which("biomcp")
assert BIOMCP_BIN, "biomcp CLI not found on PATH — install it first: pip install biomcp-cli (or biomcp-python), then restart the kernel"
print("biomcp binary:", BIOMCP_BIN)

help_text = subprocess.run([BIOMCP_BIN, "--help"], capture_output=True, text=True).stdout
biomcp_subcommand = "serve" if "serve" in help_text and "run" not in help_text.split("Commands")[-1] else "run"
print("Using subcommand:", biomcp_subcommand)

multi_server_client = MultiServerMCPClient({
    "mofa": {"command": sys.executable, "args": [str(SERVER_PATH)], "transport": "stdio"},
    "biomcp": {"command": BIOMCP_BIN, "args": [biomcp_subcommand], "transport": "stdio"},
})


### Re-bind and re-define the agent loop over the combined tool set

Nothing about `bind_tools` or `run_agent` changes conceptually — Claude just now has a larger, mixed set of tools to choose from, some fronting our own model, some fronting public biomedical databases.

In [ ]:
async def _get_all_tools():
    return await multi_server_client.get_tools()

all_tools = await _get_all_tools()
all_tools_by_name = {t.name: t for t in all_tools}

mofa_tool_names = set(tools_by_name)          # from Section 3 (mofa-only client)
biomcp_tool_names = set(all_tools_by_name) - mofa_tool_names

print(f"Loaded {len(all_tools)} tools total")
print("  from mofa   :", sorted(mofa_tool_names))
print("  from biomcp :", sorted(biomcp_tool_names))


In [ ]:
llm_with_all_tools = llm.bind_tools(all_tools)
print("Bound", len(all_tools), "tools (mofa + biomcp) to", MODEL)

#################################################
# Write the system prompt for an agent with two tool families: MOFA tools and
# BioMCP tools. Tell it when to use which, and how to handle the fact that MOFA's
# Ensembl gene IDs carry a version suffix that BioMCP tools do not recognise.
SYSTEM_MULTI = (
    #### YOUR PROMPT HERE ####
)
#################################################

async def run_agent_multi(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    messages = [SystemMessage(content=SYSTEM_MULTI), HumanMessage(content=question)]
    for step in range(max_steps):
        ai = await llm_with_all_tools.ainvoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai
        for call in ai.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = await all_tools_by_name[call["name"]].ainvoke(call["args"])
            if verbose:
                # Spot-check what actually came back over the wire -- this is the
                # evidence for the privacy claim in the closing markdown: only the
                # arguments above went OUT to BioMCP; this is what came BACK.
                preview = json.dumps(result, default=str)
                if len(preview) > 300:
                    preview = preview[:300] + f"... [{len(preview)} chars total]"
                print(f"           <- {preview}")
            messages.append(ToolMessage(content=json.dumps(result, default=str),
                                        tool_call_id=call["id"]))
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

In [ ]:
final_multi = await run_agent_multi(
    "Factor2 is associated with breast cancer subtype. What biological processes "
    "are enriched among its top 50 genes?",
    max_steps=12,
)
print(final_multi.content)


In [ ]:
#################################################
# Write a narrower, more directive question asking specifically for BioMCP's
# gene-set enrichment on Factor2's top 5 genes, and nothing else.
final_multi = await run_agent_multi(
    #### YOUR PROMPT HERE ####,
    max_steps=12,
)
print(final_multi.content)
#################################################

### Possible problems to discuss

**Is it safe to run the external server on our dataset?**

- The raw multi-omics data and the fitted MOFA model never go to BioMCP. They live only inside `mofa_mcp_server.py`'s process, loaded once from local cache. BioMCP has no access to that server or its data.
- What actually reaches BioMCP is whatever Claude puts into a tool call's arguments — in the trace above, that is Ensembl gene IDs (`ENSG00000160180`) or gene symbols. BioMCP then relays those terms out to public APIs (PubMed, ClinicalTrials.gov, MyGene.info, g:Profiler, etc.), so those third-party services see the query terms too.
- Separately, and this is true regardless of BioMCP: the full conversation, including MOFA tool results, already goes to the model provider's API as part of the agent loop. That is a standard consideration for any LLM tool-calling setup, not specific to adding BioMCP.

**Why this is low-risk for this dataset specifically:** TCGA breast-cancer data is a public, de-identified research cohort — gene names, factor loadings, and PAM50 subtype labels derived from it are not sensitive on their own. So in this notebook, the practical exposure is small.

Two things worth knowing if you are adapting this to a less public dataset:

1. **The system prompt is a guideline, not a technical boundary.** Nothing stops Claude from putting more context into a tool call than just a gene ID — it could phrase a query mentioning cohort size or subtype distribution if it thought that was helpful. The prompt discourages this but does not enforce it.
2. **You can enforce it in code.** If you want a hard guarantee, wrap the BioMCP tool calls with a validator that only allows arguments matching an expected pattern (bare Ensembl IDs, HGNC symbols, drug/disease names) and rejects anything else before it reaches `ainvoke`.

(Live demo tip: the printed tool results above are the evidence for this — you can point at exactly what went out and what came back, rather than asking the audience to take the claim on faith.)